# 07 - Analisis final

Fase 4 del TFM: consolidar resultados de clasificacion, segmentacion y explicabilidad Grad-CAM en tablas y figuras finales.

Este notebook no entrena modelos. Lee artefactos ya generados y construye la evidencia para responder RQ1-RQ5.

In [ ]:
from pathlib import Path
import subprocess
import sys

import pandas as pd
from IPython.display import Image, Markdown, display

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

FINAL_DIR = PROJECT_ROOT / 'results' / 'final_analysis'
FIGURES_DIR = FINAL_DIR / 'figures'
PYTHON_BIN = PROJECT_ROOT / '.conda' / 'bin' / 'python'
PYTHON_BIN = str(PYTHON_BIN if PYTHON_BIN.exists() else sys.executable)

PROJECT_ROOT

## Regenerar consolidado final

El script `scripts/build_final_analysis.py` recopila todos los summaries/resultados disponibles y genera:

- tablas maestras CSV;
- intervalos de confianza bootstrap para clasificacion;
- comparacion McNemar aproximada entre los dos mejores clasificadores por modalidad;
- figuras resumen;
- respuestas sinteticas a RQ1-RQ5.

In [ ]:
RUN_BUILD = True

if RUN_BUILD:
    subprocess.run(
        [PYTHON_BIN, str(PROJECT_ROOT / 'scripts' / 'build_final_analysis.py')],
        cwd=PROJECT_ROOT,
        check=True,
    )

## Cargar tablas finales

In [ ]:
classification_df = pd.read_csv(FINAL_DIR / 'classification_results_with_ci.csv')
classification_best_acc = pd.read_csv(FINAL_DIR / 'classification_best_by_accuracy.csv')
classification_best_f1 = pd.read_csv(FINAL_DIR / 'classification_best_by_f1.csv')
mcnemar_df = pd.read_csv(FINAL_DIR / 'classification_mcnemar_top2.csv')
segmentation_df = pd.read_csv(FINAL_DIR / 'segmentation_results.csv')
segmentation_best = pd.read_csv(FINAL_DIR / 'segmentation_best_by_dice.csv')
xai_df = pd.read_csv(FINAL_DIR / 'xai_gradcam_results.csv')

print('Clasificacion:', classification_df.shape)
print('Segmentacion:', segmentation_df.shape)
print('XAI:', xai_df.shape)

## Resultados principales de clasificacion

In [ ]:
cols = [
    'dataset', 'experiment', 'architecture', 'balance_strategy',
    'accuracy', 'accuracy_ci_low', 'accuracy_ci_high',
    'f1_macro', 'f1_macro_ci_low', 'f1_macro_ci_high', 'auc_roc_macro'
]
display(classification_best_acc[cols])
display(classification_best_f1[cols])
display(mcnemar_df)

## Resultados principales de segmentacion

In [ ]:
display(segmentation_best[[
    'dataset', 'experiment', 'architecture', 'variant_name', 'loss_name',
    'dice', 'iou', 'pixel_accuracy', 'threshold'
]])

display(segmentation_df[[
    'dataset', 'experiment', 'architecture', 'variant_name', 'dice', 'iou', 'pixel_accuracy', 'threshold'
]].sort_values(['dataset', 'dice'], ascending=[True, False]).head(20))

## Resultados principales de explicabilidad Grad-CAM

In [ ]:
display(xai_df[[
    'dataset', 'experiment', 'architecture', 'mask_split', 'mask_reference',
    'num_examples', 'mean_saliency_mask_iou', 'mean_saliency_inside_mask_ratio',
    'peak_inside_mask_rate'
]])

## Figuras finales

In [ ]:
for figure_name in [
    'classification_accuracy_f1_macro.png',
    'segmentation_dice_iou.png',
    'xai_gradcam_alignment.png',
]:
    figure_path = FIGURES_DIR / figure_name
    print(figure_path.relative_to(PROJECT_ROOT))
    display(Image(filename=str(figure_path)))

## Respuestas RQ1-RQ5

In [ ]:
rq_summary = (FINAL_DIR / 'rq_summary.md').read_text()
display(Markdown(rq_summary))

## Lectura para la memoria

Puntos que se pueden trasladar directamente a resultados/discusion:

- CXR tiene rendimiento alto y estable; CT es la modalidad limitante.
- DenseNet-121 + weighted CE es el mejor clasificador CXR.
- En CT, ResNet-50 gana levemente en accuracy, pero DenseNet-121 tiene mejor F1-macro/AUC.
- CXR segmenta pulmon de forma muy robusta; CT lesion mejora con experimentacion pero sigue siendo dificil.
- Grad-CAM en CXR muestra plausibilidad anatomica parcial; en CT no se alinea con lesion anotada.
- El resultado final es defendible porque incluye hallazgos positivos y limitaciones reales, no solo metricas favorables.